<a href="https://colab.research.google.com/github/sajid-shahriar/pytorch-tutorial/blob/main/ltb_automatic_differentiation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import torch

import torch.nn.functional as F

In [22]:
x = torch.ones(5)
y = torch.zeros(3)
w = torch.randn(5,3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w) + b

loss = F.binary_cross_entropy_with_logits(z, y)

print(loss)

tensor(1.1260, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


You can set the value of requires_grad when creating a tensor, or later by using x.requires_grad_(True) method.

In [12]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss= {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x783b538695a0>
Gradient function for loss= <BinaryCrossEntropyWithLogitsBackward0 object at 0x783b53856020>


In [13]:
loss.backward()
print(f"gradient of w: {w.grad}")
print(f"gradient of b: {b.grad}")

gradient of w: tensor([[0.0033, 0.0872, 0.2563],
        [0.0033, 0.0872, 0.2563],
        [0.0033, 0.0872, 0.2563],
        [0.0033, 0.0872, 0.2563],
        [0.0033, 0.0872, 0.2563]])
gradient of b: tensor([0.0033, 0.0872, 0.2563])


We can only obtain the grad properties for the leaf nodes of the computational graph, which have requires_grad property set to True. For all other nodes in our graph, gradients will not be available.

We can only perform gradient calculations using backward once on a given graph, for performance reasons. If we need to do several backward calls on the same graph, we need to pass retain_graph=True to the backward call.

#Disabling Gradient Tracking


In [14]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

True


In [15]:
# We can stop tracking computations by surrounding
# our computation code with torch.no_grad() block.

with torch.no_grad():
  z = torch.matmul(x, w) + b

print(z.requires_grad)

False


In [16]:
# other way to do that is through detach() method

z = torch.matmul(x, w) + b
z_det = z.detach()
print(z_det.requires_grad)

False


There are reasons you might want to disable gradient tracking:

* To mark some parameters
in your neural network as frozen parameters.

* To speed up computations when you are only doing forward pass, because computations on tensors that do not track gradients would be more efficient.



Conceptually, autograd keeps a record of data (tensors) and all executed operations (along with the resulting new tensors) in a directed acyclic graph (DAG) consisting of Function objects. In this DAG, leaves are the input tensors, roots are the output tensors.

#Tensor gradients and Jacobian products


In [19]:
inp = torch.eye(4,5, requires_grad=True)

out = (inp + 1).pow(2).T

print(inp)

print(out)

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.]], requires_grad=True)
tensor([[4., 1., 1., 1.],
        [1., 4., 1., 1.],
        [1., 1., 4., 1.],
        [1., 1., 1., 4.],
        [1., 1., 1., 1.]], grad_fn=<PermuteBackward0>)


In [21]:
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call: \n{inp.grad}\n")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"Second calls: \n{inp.grad}\n")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"Call after the zeroing gradients: \n{inp.grad}\n")


First call: 
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second calls: 
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after the zeroing gradients: 
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])



Notice that when we call backward for the second time with the same argument, the value of the gradient is different. This happens because when doing backward propagation, PyTorch accumulates the gradients, i.e. the value of computed gradients is added to the grad property of all leaf nodes of computational graph.

Previously we were calling backward() function without parameters. This is essentially equivalent to calling backward(torch.tensor(1.0)), which is a useful way to compute the gradients in case of a scalar-valued function, such as loss during neural network training.